# Notebook 8 – Optimization Algorithms

An optimizer uses gradients to update a model's weights. We will first see how training data is grouped, then look at common optimizers and try a few learning rates.

## 1. Gradient Descent

**Gradient descent** is a way to reduce a model's loss. It uses the gradient to decide which direction to move the weights. A batch is the group of examples used to calculate one update.

## 2. Batch Gradient Descent

Batch Gradient Descent calculates the gradient using the **whole training dataset**, then makes one update. For 100 training examples, all 100 are used for that update. It can give a steady direction, but large datasets can make each update slow and memory-heavy.

## 3. Stochastic Gradient Descent

Stochastic Gradient Descent calculates a gradient from **one training example** and updates weights straight away. With 100 examples, it can update 100 times in one pass through the data. Updates are quick but can be noisy because each example may point in a slightly different direction.

## 4. Mini-batch Gradient Descent

Mini-batch Gradient Descent uses a **small group** of examples for each update, such as 10 out of 100. It gives more frequent updates than full-batch training, while being steadier than using one example at a time. This is a common approach for neural networks.

In PyTorch, `torch.optim.SGD` can be used for these approaches. The batch style depends on how many examples were used to calculate the loss before calling `.backward()` and `optimizer.step()`. The SGD optimizer itself is explained below.

## 5. Learning Rate

The **learning rate** controls the size of each weight update. For example, suppose `w = 2` and its gradient is `−4`. Using `w_new = w − learning_rate × gradient`:
- At learning rate `0.05`, w becomes `2.2`: a small step.
- At learning rate `0.5`, w becomes `4.0`: a much larger step.

A learning rate that is too small can make learning very slow. A rate that is too large can jump past a good answer and make the loss rise. The experiment below lets us compare several learning rates.

## 6. SGD optimizer

**Concept:** plain Stochastic Gradient Descent (SGD) adjusts a weight using its current gradient. Here, “SGD” is the name of the PyTorch optimizer; the amount of data used for each update still determines whether training is stochastic, mini-batch, or full-batch.

**Mathematical intuition and update:** move opposite to the gradient: `weight = weight − learning_rate × gradient`. For `w = 2`, gradient `−4`, and learning rate `0.1`, the new weight is `2 − 0.1 × (−4) = 2.4`.

**Advantages:** simple, memory-efficient, and can work very well when tuned.
**Limitations:** may be slow or noisy and often needs learning-rate tuning.
**When to use:** a useful simple baseline, especially when training can run for enough steps.

## 7. SGD with Momentum

**Concept:** momentum keeps some of the previous update direction, like a ball gathering speed as it rolls downhill.

**Mathematical intuition and update:** `velocity = momentum × old_velocity + gradient`, then subtract `learning_rate × velocity` from the weight. If gradients keep pointing the same way, velocity builds and steps move faster in that direction.

**Advantages:** can speed up consistent progress and reduce some zig-zagging.
**Limitations:** adds a setting to tune; too much momentum can overshoot.
**When to use:** when plain SGD moves slowly or zig-zags.

## 8. RMSprop

**Concept:** RMSprop keeps an average of recent squared gradients and uses it to scale each parameter's step.

**Mathematical intuition and update:** divide the gradient by the square root of its moving average (plus a tiny value for stability), then apply the learning rate. For example, if the average is 1.6 and the gradient is −4, the scaled gradient is about −3.16; with rate 0.1, w moves from 2 to about 2.32.

**Advantages:** adapts step sizes for different parameters and can help when gradients vary.
**Limitations:** has extra settings and can be less predictable to tune.
**When to use:** a practical adaptive optimizer, often considered for recurrent or changing problems.

## 9. Adam

**Concept:** Adam combines momentum-like averaging of gradients with RMSprop-like scaling by squared gradients.

**Mathematical intuition and update:** it tracks averages of gradients and squared gradients, corrects their early bias, and uses them to choose a scaled update for each weight. On the first step of our simple example, the scaled direction is about −1, so a rate of 0.1 moves w from 2 to about 2.1.

**Advantages:** often makes useful progress with little initial tuning and adapts to different parameters.
**Limitations:** uses more memory than basic SGD; carefully tuned SGD may perform better on some tasks.
**When to use:** a convenient starting choice for many neural-network experiments.

## 10. Compare SGD and Adam

Both optimizers use gradients to reduce loss, but they choose update sizes differently.

| | SGD | Adam |
|---|---|---|
| How the step is chosen | Uses the current gradient (and optional momentum) | Adapts steps using averages of gradients and squared gradients |
| Settings to start with | Learning rate; momentum is optional | Learning rate; default averaging settings are often a useful start |
| Strength | Simple, memory-efficient, and can perform well when tuned | Often makes useful progress quickly with less tuning |
| Limitation | May need careful tuning and more training steps | Uses more memory and is not always best for final results |

**Simple takeaway:** Adam is a convenient first try; SGD is a simple option that can work very well with tuning. There is no best choice for every problem, so compare loss on data the model did not train on.

## 11. Experiment: different learning rates

We will teach a single number `w` to approach 3 by minimizing `(w − 3)²`. Each trial starts from the same value. Run the code and compare the printed losses: a very small rate may make slow progress, while a rate that is too large may jump past the answer. The loss values show the effect during training.

In [1]:
import torch

def train_with_sgd(learning_rate):
    w = torch.nn.Parameter(torch.tensor(0.0))
    optimizer = torch.optim.SGD([w], lr=learning_rate)
    losses = []

    for step in range(10):
        optimizer.zero_grad()
        loss = (w - 3) ** 2
        loss.backward()
        optimizer.step()
        losses.append(round(loss.item(), 4))

    return w.item(), losses

for rate in [0.01, 0.1, 0.8, 1.1]:
    final_w, losses = train_with_sgd(rate)
    print(f"Learning rate {rate}: final w={final_w:.3f}, losses={losses}")

Learning rate 0.01: final w=0.549, losses=[9.0, 8.6436, 8.3013, 7.9726, 7.6569, 7.3537, 7.0625, 6.7828, 6.5142, 6.2562]
Learning rate 0.1: final w=2.678, losses=[9.0, 5.76, 3.6864, 2.3593, 1.5099, 0.9664, 0.6185, 0.3958, 0.2533, 0.1621]
Learning rate 0.8: final w=2.982, losses=[9.0, 3.24, 1.1664, 0.4199, 0.1512, 0.0544, 0.0196, 0.0071, 0.0025, 0.0009]
Learning rate 1.1: final w=-15.575, losses=[9.0, 12.96, 18.6624, 26.8739, 38.6984, 55.7257, 80.2449, 115.5527, 166.396, 239.6102]


**Notice:** a small learning rate usually reduces loss slowly. A moderate one should move toward 3 faster. If the rate is too large, the weight may bounce past the answer and the loss may increase. The exact behavior depends on the problem and number of steps.

## 12. Compare SGD and Adam on the same tiny task

Both optimizers start from the same value and try to make `w` approach 3. The learning rates do not necessarily have the same best value for each optimizer, so treat this as a small illustration—not a benchmark.

In [2]:
def compare_optimizers(optimizer_name):
    w = torch.nn.Parameter(torch.tensor(0.0))

    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD([w], lr=0.1)
    else:
        optimizer = torch.optim.Adam([w], lr=0.1)

    for step in range(20):
        optimizer.zero_grad()
        loss = (w - 3) ** 2
        loss.backward()
        optimizer.step()

    return w.item(), loss.item()

for name in ["SGD", "Adam"]:
    final_w, final_loss = compare_optimizers(name)
    print(f"{name}: w={final_w:.3f}, loss={final_loss:.4f}")

SGD: w=2.965, loss=0.0019
Adam: w=1.881, loss=1.4443


The optimizer changes the path the weight takes. In a real task, compare training and validation loss, and try reasonable learning rates. Next, we will build a small PyTorch network whose parameters an optimizer can update.